# Scraping Data X (Twitter): Kebijakan WFH ASN Hari Jumat

Notebook ini mengambil data **historis** dari X **tanpa API** menggunakan `snscrape`, untuk keperluan **analisis sentimen**.

**Cara pakai:** jalankan tiap sel berurutan (Shift+Enter) dari atas ke bawah.

> ⚠️ **Catatan jujur:** IP Google Colab adalah IP datacenter Google, yang **sering diblokir X**. Kalau sel scraping error `blocked`/`4 requests failed`, itu wajar — coba: kurangi jumlah, persempit tanggal, jalankan ulang beberapa saat kemudian, atau pakai runtime lokal.

## 1. Install snscrape (versi terbaru dari GitHub)

In [ ]:
!pip install -q git+https://github.com/JustAnotherArchivist/snscrape.git
print('snscrape terpasang. Restart runtime bila diminta, lalu lanjut ke sel berikutnya.')

## 2. Konfigurasi pencarian

Ubah `MAX_TWEETS`, rentang tanggal (`since`/`until`), atau kata kunci sesuai kebutuhan.

In [ ]:
MAX_TWEETS = 2000
OUTPUT_FILE = 'tweets_wfh_asn_historis.csv'

# Operator sama seperti 'Pencarian Lanjutan' di web X.
QUERY = (
    '("WFH ASN" OR "WFH PNS" OR "ASN WFH" OR "kerja dari rumah ASN" '
    'OR (ASN Jumat WFH) OR (PNS Jumat "work from home")) '
    'lang:id since:2024-01-01 until:2025-12-31 -filter:retweets'
)
print('Query:', QUERY)

## 3. Jalankan scraping

In [ ]:
import itertools
import pandas as pd
import snscrape.modules.twitter as sntwitter

rows = []
scraper = sntwitter.TwitterSearchScraper(QUERY)

try:
    for i, tweet in enumerate(itertools.islice(scraper.get_items(), MAX_TWEETS)):
        rows.append({
            'id': tweet.id,
            'created_at': tweet.date,
            'username': tweet.user.username,
            'likes': tweet.likeCount,
            'retweets': tweet.retweetCount,
            'replies': tweet.replyCount,
            'text': tweet.rawContent.replace('\n', ' '),
        })
        if (i + 1) % 100 == 0:
            print(f'  ... {i + 1}/{MAX_TWEETS} tweet terkumpul')
except Exception as e:
    print(f'[!] Berhenti karena error: {e}')
    print('    Kemungkinan X memblokir akses (umum di Colab). Data yang sudah terambil tetap disimpan.')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'\nSelesai. {len(df)} tweet tersimpan di {OUTPUT_FILE}')
df.head()

## 4. Unduh file CSV ke komputer

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## 5. (Opsional) Analisis sentimen langsung di Colab

Klasifikasi positif / netral / negatif memakai model IndoBERT.

In [ ]:
!pip install -q transformers torch

import re
from transformers import pipeline

LABEL_MAP = {'LABEL_0': 'positif', 'LABEL_1': 'netral', 'LABEL_2': 'negatif'}

def bersihkan(t):
    t = re.sub(r'http\S+|www\.\S+', '', str(t))
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_text'] = df['text'].apply(bersihkan)
df = df[df['clean_text'].str.len() > 3].reset_index(drop=True)

clf = pipeline('sentiment-analysis',
               model='mdhugol/indonesia-bert-sentiment-classification',
               truncation=True, max_length=256)

hasil = clf(df['clean_text'].tolist())
df['sentimen'] = [LABEL_MAP.get(r['label'], r['label']) for r in hasil]
df['skor'] = [round(r['score'], 3) for r in hasil]

df.to_csv('tweets_wfh_asn_sentimen.csv', index=False, encoding='utf-8')
print(df['sentimen'].value_counts())
df[['text', 'sentimen', 'skor']].head(10)

In [ ]:
import matplotlib.pyplot as plt

ringkasan = df['sentimen'].value_counts().reindex(['positif', 'netral', 'negatif'])
ringkasan.plot(kind='bar', color=['#2ecc71', '#95a5a6', '#e74c3c'])
plt.title('Sentimen Publik: Kebijakan WFH ASN Hari Jumat')
plt.ylabel('Jumlah Tweet')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentimen_wfh_asn.png', dpi=120)
plt.show()